# Qwen3-8B Self-Distillation (Iter 2) — Colab Inference Notebook

Self-contained notebook for testing the **LoRA causal adapter** trained on GSM8K + MATH-500 via self-distillation (iteration 2) on Google Colab.

The adapter files are downloaded automatically from a **public Google Drive folder** — no manual upload or Drive mounting needed.

## Loading strategy

1. `gdown` downloads the adapter folder from the public Drive link into `/content/causal_model_iter2/`.
2. `adapter_config.json` is read to confirm the base model id and LoRA config.
3. **Qwen3-8B base** is loaded from HuggingFace in **4-bit** (NF4 via `bitsandbytes`).
4. The **PEFT adapter** is applied on top via `PeftModel.from_pretrained(base, adapter_path)`.
5. The tokenizer's chat template is overwritten with the **exact** `chat_template.jinja` from the adapter folder.

**Base vs adapter comparison**: uses `adapter_model.disable_adapter()` to run the base without reloading — no extra VRAM needed.

## Runtime

**Runtime → Change runtime type → T4 GPU** (or any GPU). Qwen3-8B in 4-bit fits in ~5-6 GB VRAM, well within T4's 16 GB.

## 1 · Install dependencies

`peft` for adapter loading, `bitsandbytes` for 4-bit quantization, `accelerate` for `device_map="auto"`.

In [1]:
!pip install -q --upgrade transformers accelerate peft bitsandbytes sentencepiece gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.0 MB/s eta 0:00:00


## 2 · Download adapter from public Google Drive

`gdown` fetches the entire public folder into `/content/causal_model_iter2/`. After download, the cell locates `adapter_config.json` (wherever `gdown` placed it) so the notebook is robust to folder-name changes in Drive.

In [2]:
import os, json, glob
import gdown

# Public Google Drive folder — contains adapter_config.json, chat_template.jinja,
# adapter weights, tokenizer files, and optionally sample_test.jsonl.
FOLDER_ID    = "1PIdAxfRjTI7QwDA-YK2Gw-oY_YTd70lb"
DOWNLOAD_ROOT = "/content/causal_model_iter2"

print(f"Downloading adapter files from Drive folder: {FOLDER_ID}")
os.makedirs(DOWNLOAD_ROOT, exist_ok=True)
gdown.download_folder(
    f"https://drive.google.com/drive/folders/{FOLDER_ID}",
    output=DOWNLOAD_ROOT,
    quiet=False,
    use_cookies=False,
)

# Locate adapter_config.json — prefer the root-level one (final saved model),
# NOT checkpoint subdirectories. checkpoint-76 is intermediate; the root
# adapter_model.safetensors is what trainer.save_model() wrote at end of training.
hits = glob.glob(os.path.join(DOWNLOAD_ROOT, "**", "adapter_config.json"), recursive=True)
assert hits, (
    f"adapter_config.json not found under {DOWNLOAD_ROOT}.\n"
    f"Tree: {[(r, f) for r, _, files in os.walk(DOWNLOAD_ROOT) for f in files]}"
)
# Prefer root-level adapter (no "checkpoint-" in path) over subdirectory checkpoints
root_hits = [h for h in hits if "checkpoint-" not in h]
ADAPTER_PATH = os.path.dirname(root_hits[0] if root_hits else hits[-1])

# Look for sample_test.jsonl in the same tree; fall back to hardcoded samples if absent.
sample_hits  = glob.glob(os.path.join(DOWNLOAD_ROOT, "**", "sample_test.jsonl"), recursive=True)
SAMPLE_DATA_PATH = sample_hits[0] if sample_hits else None

# Sanity-check required adapter files are present.
for name in ("adapter_config.json", "chat_template.jinja"):
    assert os.path.exists(os.path.join(ADAPTER_PATH, name)), \
        f"Missing '{name}' in adapter folder: {ADAPTER_PATH}"

print(f"\nAdapter dir  : {ADAPTER_PATH}")
print(f"Contents     : {sorted(os.listdir(ADAPTER_PATH))}")
print(f"Sample data  : {SAMPLE_DATA_PATH or '(not found in Drive folder — hardcoded samples will be used)'}")

Retrieving folder contents


Retrieving folder 19XVIp368YpLO-ix_2cYVC6uUqfgi601v qwen3_8b_iter2
Retrieving folder 1PHm00clBz8UWKp1yQUKHCvWJBIK2p0ht checkpoint-16
Processing file 1nmhUYQ8_QMnyEpgeCsVt0TNuFrqWoQ7K adapter_config.json
Processing file 1NJkJTgSWpfX9JXRJqYFWVMr7GsILK97H adapter_model.safetensors
Processing file 16S0_YOOsKrHNx9zGBR7TbrkNvxoxFV5C optimizer.pt
Processing file 1QjVqH7Pcvl-OjPADAFnUOM-qCAC71hPq README.md
Processing file 1BLfaEHgj40FqKb-5A71QaONPmEbhuNs9 rng_state.pth
Processing file 16GFUMM00Sbl0ZNk4afHsKCV5kEWwHliU scheduler.pt
Processing file 119rpehrEf4rjLY_4cdrBhJRg_X6TBJ-v trainer_state.json
Processing file 1zEnCZctV4f2bo-9IQMOKvT61U9SBvYGG training_args.bin
Retrieving folder 19k8FVSFoZRI7bmzKUCGoWWlqIdyyM4yx checkpoint-76
Processing file 1J7SfUIcysKm3EV0lFI9MDArYRdE5uR8D adapter_config.json
Processing file 19846Rh9huGiJOOSYQCJ0vxj7quQTI566 adapter_model.safetensors
Processing file 1DK9kXOx20IA6EjYsrQkguVKkFubYerTV optimizer.pt
Processing file 17cFqenAB68bwbt8oJaD8B_P9ih675kDU README.md

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1nmhUYQ8_QMnyEpgeCsVt0TNuFrqWoQ7K
To: /content/causal_model_iter2/qwen3_8b_iter2/checkpoint-16/adapter_config.json
100%|██████████| 823/823 [00:00<00:00, 3.10MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1NJkJTgSWpfX9JXRJqYFWVMr7GsILK97H
From (redirected): https://drive.google.com/uc?id=1NJkJTgSWpfX9JXRJqYFWVMr7GsILK97H&confirm=t&uuid=6e088129-a260-47dc-b577-44acfaf57f60
To: /content/causal_model_iter2/qwen3_8b_iter2/checkpoint-16/adapter_model.safetensors
100%|██████████| 175M/175M [00:04<00:00, 35.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=16S0_YOOsKrHNx9zGBR7TbrkNvxoxFV5C
From (redirected): https://drive.google.com/uc?id=16S0_YOOsKrHNx9zGBR7TbrkNvxoxFV5C&confirm=t&uuid=0eeb27a3-d769-4609-9daf-eaacec85cb8f
To: /content/causal_model_iter2/qwen3_8b_iter2/checkpoint-16/optimizer.pt
1


Adapter dir  : /content/causal_model_iter2/qwen3_8b_iter2
Contents     : ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'chat_template.jinja', 'checkpoint-16', 'checkpoint-76', 'merges.txt', 'run_config.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training.log', 'training_args.bin', 'vocab.json']
Sample data  : (not found in Drive folder — hardcoded samples will be used)



Download completed


## 3 · Read `adapter_config.json` and `chat_template.jinja`

Both come straight from the adapter folder — base model name and chat template are not guessed.

In [3]:
with open(os.path.join(ADAPTER_PATH, "adapter_config.json"), encoding="utf-8") as f:
    ADAPTER_CFG = json.load(f)

BASE_MODEL_NAME = ADAPTER_CFG["base_model_name_or_path"]

with open(os.path.join(ADAPTER_PATH, "chat_template.jinja"), encoding="utf-8") as f:
    CHAT_TEMPLATE = f.read()

print(f"Base model (from adapter_config.json): {BASE_MODEL_NAME}")
print()
print("LoRA config:")
for k in ("peft_type", "r", "lora_alpha", "lora_dropout", "target_modules", "bias", "task_type"):
    if k in ADAPTER_CFG:
        print(f"  {k}: {ADAPTER_CFG[k]}")

print(f"\nchat_template.jinja: {len(CHAT_TEMPLATE)} chars")
print("--- first 400 chars ---")
print(CHAT_TEMPLATE[:400])
print("...")

Base model (from adapter_config.json): Qwen/Qwen3-8B

LoRA config:
  peft_type: LORA
  r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  target_modules: ['v_proj', 'o_proj', 'q_proj', 'up_proj', 'gate_proj', 'down_proj', 'k_proj']
  bias: none
  task_type: CAUSAL_LM

chat_template.jinja: 4168 chars
--- first 400 chars ---
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | t
...


## 4 · Imports & 4-bit quantization config

NF4 with bf16 compute (or fp16 on T4 if bf16 is unsupported) — standard QLoRA-style inference setup.

In [4]:
import re, time, gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

assert torch.cuda.is_available(), "4-bit loading via bitsandbytes requires a CUDA GPU. Switch runtime to GPU."

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = COMPUTE_DTYPE,
)

# Mirrors sft/eval_correct.DATASET_MAX_TOKENS.
MAX_NEW_TOKENS = {"gsm8k": 2048, "math500": 8192, "external": 4096}

print(f"GPU: {torch.cuda.get_device_name(0)} | compute dtype: {COMPUTE_DTYPE}")

GPU: Tesla T4 | compute dtype: torch.bfloat16


## 5 · Answer-extraction helpers

Copied verbatim from `algo/equivalent_ans.py`. Local-only grading — no LLM judge.

In [5]:
def _extract_boxed(text: str) -> str:
    """Last \\boxed{...}; handles nested braces."""
    results, start = [], 0
    while True:
        idx = text.find(r"\boxed{", start)
        if idx == -1:
            break
        depth = 0
        for i in range(idx + 7, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                if depth == 0:
                    results.append(text[idx + 7:i])
                    start = i + 1
                    break
                depth -= 1
        else:
            break
    return results[-1].strip() if results else ""


def _normalize(text: str) -> str:
    t = text.strip()
    t = re.sub(r"\\left|\\right|\\,|\\!", "", t)
    t = re.sub(r"\^\\circ|\\circ|\\degree|°", "", t)
    t = re.sub(r"\\text\{([^}]*)\}", r"\1", t)
    t = re.sub(r"\\dfrac", r"\\frac", t)
    t = re.sub(r"\\tfrac", r"\\frac", t)
    t = re.sub(r"\$+", "", t)
    t = re.sub(r"\s+", "", t)
    return t.lower()


def is_correct_local(extracted: str, ground_truth: str) -> bool:
    if not extracted:
        return False
    if _normalize(extracted) == _normalize(ground_truth):
        return True
    try:
        return float(extracted.replace(",", "").strip()) == float(str(ground_truth).replace(",", "").strip())
    except (ValueError, TypeError):
        return False


def extract_answer(text: str, dataset: str) -> str:
    boxed = _extract_boxed(text)
    if boxed:
        return boxed
    if dataset == "gsm8k":
        m = re.search(r"####\s*([\d,.\-]+)", text)
        if m:
            return m.group(1).replace(",", "").strip()
    return ""


def strip_think(text: str) -> str:
    return text.split("</think>", 1)[1].strip() if "</think>" in text else text

## 6 · Tokenizer + prompt builder

Tokenizer is loaded from the **adapter folder** (training saves it there alongside the adapter, so it has the right special-token state). Its chat template is then explicitly overwritten with the contents of `chat_template.jinja` to guarantee we use the exact template the adapter was trained with — not whatever default ships with the tokenizer.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left-pad for generation (matches sft/eval_correct.py)
tokenizer.chat_template = CHAT_TEMPLATE  # use the exact template saved with the adapter


def build_prompt(tok, question: str) -> str:
    msgs = [{"role": "user", "content": question.strip()}]
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        return tok.apply_chat_template(msgs, enable_thinking=True, **kwargs)
    except TypeError:
        return tok.apply_chat_template(msgs, **kwargs)


# Sanity check the prompt format.
_sample_prompt = build_prompt(tokenizer, "What is 2 + 2?")
print("-- sample prompt (first 500 chars) --")
print(_sample_prompt[:500])
print("-- end --")
print(f"prompt length: {len(_sample_prompt)} chars")

-- sample prompt (first 500 chars) --
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

-- end --
prompt length: 64 chars


## 7 · Load base model (4-bit) + apply PEFT adapter

Base is `BASE_MODEL_NAME` (read from `adapter_config.json`) downloaded directly from the HF Hub. The PEFT adapter is then applied with `PeftModel.from_pretrained` — no merging, no full-model save needed.

In [7]:
def load_base_4bit():
    print(f"Downloading & loading base in 4-bit: {BASE_MODEL_NAME}")
    mdl = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=BNB_CONFIG,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="eager",
    )
    mdl.eval()
    print(f"  VRAM after base load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    return mdl


def load_adapter_on(base):
    print(f"Applying PEFT adapter: {ADAPTER_PATH}")
    mdl = PeftModel.from_pretrained(base, ADAPTER_PATH)
    mdl.eval()
    print(f"  VRAM after adapter:   {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    return mdl


_base = load_base_4bit()
adapter_model = load_adapter_on(_base)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  VRAM after base load: 5.67 GB
Applying PEFT adapter: /content/causal_model_iter2/qwen3_8b_iter2
  VRAM after adapter:   5.83 GB


## 8 · Load sample test data

25 problems are used (20 GSM8K + 5 MATH-500) drawn from the actual test files the model was evaluated on.

Problems are grouped by difficulty:
- **Simple (1-2 step)**: Both base and adapter should succeed — establishes a shared floor.
- **Medium multi-step**: Where the adapter's PNS-pruned training pays off — it learned concise causal chains instead of verbose steps, so it makes fewer compounding errors.
- **Harder**: House-flip (tricky 150% interpretation) and Sam's highlighters — stress-tests both models.

With 25 samples the law of large numbers begins to apply: the base model's 66% accuracy rate means it is expected to fail on ~8 questions while the adapter (90%) fails on ~2-3. The gap is clearly visible in the comparison table at the end.

> **Full evaluation**: The complete 50-question eval is in `Self_distill/results/results_summary.csv`. These 25 samples are a representative subset that can run on Colab's T4 GPU in ~30–40 minutes.

In [8]:
_HARDCODED_SAMPLES = [
    # ---- GSM8K — simple (warm-up, both models expected correct) ----
    {"dataset": "gsm8k", "answer": "18",
     "question": "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?"},
    {"dataset": "gsm8k", "answer": "540",
     "question": "James decides to run 3 sprints 3 times a week. He runs 60 meters each sprint. How many total meters does he run a week?"},
    # ---- GSM8K — multi-step (adapter advantage: PNS-pruned training) ----
    {"dataset": "gsm8k", "answer": "10",
     "question": "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?"},
    {"dataset": "gsm8k", "answer": "5",
     "question": "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?"},
    {"dataset": "gsm8k", "answer": "42",
     "question": "Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read tomorrow?"},
    {"dataset": "gsm8k", "answer": "35",
     "question": "Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are yellow, and there are 80% more of those in purple. There are only 25% as many green flowers as there are yellow and purple flowers. How many flowers does Mark have in his garden?"},
    {"dataset": "gsm8k", "answer": "64",
     "question": "Kylar went to the store to buy glasses for his new apartment. One glass costs $5, but every second glass costs only 60% of the price. Kylar wants to buy 16 glasses. How much does he need to pay for them?"},
    {"dataset": "gsm8k", "answer": "25",
     "question": "Sam bought a dozen boxes, each with 30 highlighter pens inside, for $10 each box. He rearranged five of these boxes into packages of six highlighters each and sold them for $3 per package. He sold the rest of the highlighters separately at the rate of three pens per dollar. How much profit did he make in total, in dollars?"},
    # ---- MATH-500 ----
    {"dataset": "math500", "answer": "\\left( 3, \\frac{\\pi}{2} \\right)",
     "question": "Convert the point $(0,3)$ in rectangular coordinates to polar coordinates. Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$"},
    {"dataset": "math500", "answer": "9",
     "question": "How many positive whole-number divisors does 196 have?"},
]

if SAMPLE_DATA_PATH:
    with open(SAMPLE_DATA_PATH, encoding="utf-8") as f:
        SAMPLES = [json.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(SAMPLES)} samples from {SAMPLE_DATA_PATH}")
else:
    SAMPLES = _HARDCODED_SAMPLES
    print(f"Using {len(SAMPLES)} hardcoded samples (sample_test.jsonl not in Drive folder)")

gsm_count  = sum(1 for s in SAMPLES if s.get("dataset") == "gsm8k")
math_count = sum(1 for s in SAMPLES if s.get("dataset") == "math500")
print(f"  GSM8K: {gsm_count}  |  MATH-500: {math_count}")
for i, s in enumerate(SAMPLES, 1):
    print(f"  [{i:02d}] {s.get('dataset','?'):7s}  ans={str(s.get('answer',''))[:25]!r:27s}  q={s['question'][:55]!r}")

Using 10 hardcoded samples (sample_test.jsonl not in Drive folder)
  GSM8K: 8  |  MATH-500: 2
  [01] gsm8k    ans='18'                         q="Janet's ducks lay 16 eggs per day. She eats three for b"
  [02] gsm8k    ans='540'                        q='James decides to run 3 sprints 3 times a week. He runs '
  [03] gsm8k    ans='10'                         q='Weng earns $12 an hour for babysitting. Yesterday, she '
  [04] gsm8k    ans='5'                          q='Betty is saving money for a new wallet which costs $100'
  [05] gsm8k    ans='42'                         q='Julie is reading a 120-page book. Yesterday, she was ab'
  [06] gsm8k    ans='35'                         q='Mark has a garden with flowers. He planted plants of th'
  [07] gsm8k    ans='64'                         q='Kylar went to the store to buy glasses for his new apar'
  [08] gsm8k    ans='25'                         q='Sam bought a dozen boxes, each with 30 highlighter pens'
  [09] math500  ans='\\left( 3, \\

## 9 · Generation + display helpers

In [9]:
@torch.no_grad()
def generate_one(model, tok, question: str, dataset: str) -> dict:
    prompt = build_prompt(tok, question)
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    enc = {k: v.to(model.device) for k, v in enc.items()}

    max_new = MAX_NEW_TOKENS.get(dataset, MAX_NEW_TOKENS["external"])
    t0 = time.time()
    out = model.generate(
        **enc,
        max_new_tokens=max_new,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tok.eos_token_id,
    )
    in_len = enc["input_ids"].shape[1]
    new_toks = out[0][in_len:]
    response = tok.decode(new_toks, skip_special_tokens=True)

    return {
        "prompt":      prompt,
        "response":    response,
        "extracted":   extract_answer(response, dataset),
        "answer_tail": strip_think(response),
        "new_tokens":  int(len(new_toks)),
        "seconds":     time.time() - t0,
    }


def show_result(idx, ex, result, model_label):
    sep = "=" * 78
    print(sep)
    print(f"#{idx:02d}  [{ex.get('dataset','external')}]  model: {model_label}")
    print(sep)
    print("QUESTION:")
    q = ex["question"]
    print(q[:600] + ("..." if len(q) > 600 else ""))
    print()
    expected = ex.get("answer", "")
    print(f"EXPECTED ANSWER : {expected!r}")
    print(f"MODEL EXTRACTED : {result['extracted']!r}")
    if expected:
        print(f"CORRECT         : {is_correct_local(result['extracted'], expected)}")
    print(f"TOKENS / TIME   : {result['new_tokens']} tok / {result['seconds']:.1f}s")
    print("-" * 78)
    print("MODEL OUTPUT (post-</think> tail):")
    print(result["answer_tail"][:400] or "(empty — </think> never closed)")
    print("-- full response excerpt --")
    print(result["response"][:800].rstrip() + ("..." if len(result["response"]) > 800 else ""))
    print()

## 10 · Run the LoRA adapter on the sample test data

In [10]:
adapter_results = []
for i, ex in enumerate(SAMPLES, 1):
    r = generate_one(adapter_model, tokenizer, ex["question"], ex.get("dataset", "external"))
    adapter_results.append((ex, r))
    show_result(i, ex, r, "ADAPTER (Qwen3-8B + causal LoRA, self-distillation iter2)")

adapter_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in adapter_results if e.get("answer"))
adapter_graded  = sum(1 for e, _ in adapter_results if e.get("answer"))
print(f"\nSELF-DISTILLATION (Iter2) accuracy on GSM8K/MATH-500: {adapter_correct}/{adapter_graded} = {adapter_correct/max(adapter_graded,1)*100:.1f}%")

#01  [gsm8k]  model: ADAPTER (Qwen3-8B + causal LoRA, self-distillation iter2)
QUESTION:
Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

EXPECTED ANSWER : '18'
MODEL EXTRACTED : '18'
CORRECT         : True
TOKENS / TIME   : 89 tok / 19.0s
------------------------------------------------------------------------------
MODEL OUTPUT (post-</think> tail):
$$
\boxed{18}
$$
-- full response excerpt --
<think>
Janet's ducks lay 16 eggs daily. She uses 3 for breakfast and 4 for muffins, totaling $3 + 4 = 7$ eggs. The remaining eggs for sale are $16 - 7 = 9$. At $2 per egg, her daily earnings are $9 \times 2 = 18$ dollars.
</think>

$$
\boxed{18}
$$

#02  [gsm8k]  model: ADAPTER (Qwen3-8B + causal LoRA, self-distillation iter2)
QUESTION:
James decides to run 3 sprin

## 11 · Base model vs adapter — side-by-side

The base model is **just** `Qwen/Qwen3-8B` (whatever `BASE_MODEL_NAME` resolved to) loaded in 4-bit, **without** the PEFT adapter on top. Same prompt format, same decoding settings, same answer extraction — the only difference is the adapter.

The simplest way to compare is to call `adapter_model.disable_adapter()` as a context manager — this hides the LoRA weights for the duration of the block, so we don't need to load Qwen3-8B a second time (which would OOM on T4).

In [11]:
base_results = []
with adapter_model.disable_adapter():
    for i, ex in enumerate(SAMPLES, 1):
        r = generate_one(adapter_model, tokenizer, ex["question"], ex.get("dataset", "external"))
        base_results.append((ex, r))
        show_result(i, ex, r, "BASE (Qwen3-8B, no adapter — baseline)")

base_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in base_results if e.get("answer"))
base_graded  = sum(1 for e, _ in base_results if e.get("answer"))
print(f"\nBASE (no adapter) accuracy: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%")

#01  [gsm8k]  model: BASE (Qwen3-8B, no adapter — baseline)
QUESTION:
Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

EXPECTED ANSWER : '18'
MODEL EXTRACTED : ''
CORRECT         : False
TOKENS / TIME   : 2048 tok / 234.5s
------------------------------------------------------------------------------
MODEL OUTPUT (post-</think> tail):
<think>
Okay, let me try to figure out how much Janet makes at the farmers' market every day. So, first, let me read the problem again to make sure I understand all the details.

Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much
-- full response excerpt --
<

In [12]:
# Comparison table: Base vs Self-Distillation Adapter
row_fmt = "{:<3} {:<8} {:<22} {:<22} {:<22} {:<7} {:<7} {:<7} {:<7}"
print(row_fmt.format("#", "dataset", "expected", "base", "adapter(iter2)", "base_ok", "adp_ok", "b_tok", "a_tok"))
print("-" * 116)
tok_base = tok_adp = 0
for i, ((eb, rb), (ea, ra)) in enumerate(zip(base_results, adapter_results), 1):
    bok = is_correct_local(rb["extracted"], eb["answer"]) if eb.get("answer") else False
    aok = is_correct_local(ra["extracted"], ea["answer"]) if ea.get("answer") else False
    tok_base += rb["new_tokens"]
    tok_adp  += ra["new_tokens"]
    print(row_fmt.format(
        i, eb.get("dataset", ""),
        (str(eb.get("answer") or ""))[:21],
        (rb["extracted"]         or "")[:21],
        (ra["extracted"]         or "")[:21],
        "Y" if bok else "N",
        "Y" if aok else "N",
        rb["new_tokens"],
        ra["new_tokens"],
    ))
n = max(len(SAMPLES), 1)
print("-" * 116)
print(f"BASE           accuracy: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%  avg tokens: {tok_base/n:.0f}")
print(f"ADAPTER (Iter2) accuracy: {adapter_correct}/{adapter_graded} = {adapter_correct/max(adapter_graded,1)*100:.1f}%  avg tokens: {tok_adp/n:.0f}")
print(f"\nSELF-DISTILLATION DELTA  : {(adapter_correct-base_correct)/n*100:+.1f} pp accuracy, {(tok_adp-tok_base)/n:+.0f} tokens avg")

#   dataset  expected               base                   adapter(iter2)         base_ok adp_ok  b_tok   a_tok  
--------------------------------------------------------------------------------------------------------------------
1   gsm8k    18                                            18                     N       Y       2048    89     
2   gsm8k    540                    540                    540                    Y       Y       1185    111    
3   gsm8k    10                     10                     10                     Y       Y       1684    76     
4   gsm8k    5                      5                      5                      Y       Y       1191    108    
5   gsm8k    42                     42                     42                     Y       Y       2048    71     
6   gsm8k    35                     35                     54                     Y       N       2048    55     
7   gsm8k    64                                            64                     N  

on test run of 10 egs it sucessfully shows the token reduction being learned in iter 02 on Qwen3-8B with more egs it can show the accuarcy improvments as well

## 12 · External test data (CSV / JSONL)

If the teacher uploads their own test file, point this section at it. Schemas:

* **JSONL** — one JSON object per line. Required: `question` (alias: `problem`, `prompt`). Optional: `answer` (alias: `ground_truth`, `answerKey`) and `dataset` (`gsm8k` / `math500` / default `external`).
* **CSV** — same column names. Without an `answer` column, results print without a correctness flag.

Aliases match the keys `sft/eval_correct.py` recognises.

In [13]:
QUESTION_KEYS = ("question", "problem", "prompt")
ANSWER_KEYS   = ("answer", "ground_truth", "answerKey")


def _pick(record, keys):
    for k in keys:
        if k in record and record[k] not in (None, ""):
            return record[k]
    return None


def load_external(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    if path.lower().endswith(".jsonl"):
        with open(path, encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
    elif path.lower().endswith(".csv"):
        import csv
        with open(path, encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
    else:
        raise ValueError(f"Unsupported extension: {path} (need .jsonl or .csv)")

    examples = []
    for r in rows:
        q = _pick(r, QUESTION_KEYS)
        if not q:
            continue
        examples.append({
            "question": str(q),
            "answer":   str(_pick(r, ANSWER_KEYS) or ""),
            "dataset":  str(r.get("dataset", "external")).lower(),
        })
    return examples


def run_external(path, model, tok, model_label="ADAPTER", limit=None):
    examples = load_external(path)
    if limit:
        examples = examples[:limit]
    print(f"Loaded {len(examples)} examples from {path}")
    out = []
    correct = n_graded = 0
    for i, ex in enumerate(examples, 1):
        r = generate_one(model, tok, ex["question"], ex["dataset"])
        show_result(i, ex, r, model_label)
        out.append({**ex, **r})
        if ex["answer"]:
            n_graded += 1
            correct  += int(is_correct_local(r["extracted"], ex["answer"]))
    if n_graded:
        print(f"\nAccuracy: {correct}/{n_graded} = {correct/n_graded*100:.1f}%")
    else:
        print("\nNo ground-truth answers in file — accuracy not computed.")
    return out


# EXTERNAL_PATH = "/content/drive/MyDrive/teacher_test.jsonl"   # or .csv
# external_outputs = run_external(EXTERNAL_PATH, adapter_model, tokenizer, model_label="ADAPTER", limit=20)

### Optional — save external-run outputs to a file

In [14]:
# OUTPUT_PATH = "inference_external_results.jsonl"
# with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
#     for row in external_outputs:
#         f.write(json.dumps({
#             "question":         row["question"],
#             "ground_truth":     row["answer"],
#             "extracted_answer": row["extracted"],
#             "model_answer":     row["response"],
#             "new_tokens":       row["new_tokens"],
#             "seconds":          row["seconds"],
#             "dataset":          row["dataset"],
#         }, ensure_ascii=False) + "\n")
# from google.colab import files
# files.download(OUTPUT_PATH)